# 1. Azure Network Security

This notebook covers the **Azure network security** services on the SC-900 exam. We simulate each service in Python so you can learn *how they behave* without spinning up real Azure resources.

## The services at a glance

| Service | What it protects | Layer |
|---------|-----------------|-------|
| **Azure DDoS Protection** | Public IPs from volumetric attacks | Network (L3/L4) |
| **Azure Firewall** | VNets — centralized traffic filtering | Network (L3-L7) |
| **Web Application Firewall (WAF)** | Web apps from OWASP attacks | Application (L7) |
| **Network Security Groups (NSGs)** | Subnets and NICs — port/IP filtering | Network (L3/L4) |
| **Azure Virtual Network (VNet)** | Network isolation and segmentation | Network |
| **Azure Bastion** | Secure RDP/SSH without public IPs on VMs | Access |

Each section follows a **bad practice → best practice** story so you can see *why* each control exists.

## 🔧 Setup (run this once)

Before running the code cells, make sure you've installed dependencies and selected the right kernel:

```bash
cd security-certs/sc-900/03-azure-security-solutions
uv sync
```

Then in **VS Code**: click the kernel picker in the top-right of this notebook and pick the `.venv` interpreter for this folder. If it doesn't show up, reload the window (`Cmd+Shift+P` → *Developer: Reload Window*).

> ℹ️ **No Azure account needed.** These notebooks *simulate* Azure services in pure Python so you can learn the concepts without any cloud cost.

---
## 1. Virtual Networks (VNets) and Subnets

A **VNet** is your private network in Azure. Resources inside a VNet can talk to each other by default. Resources in *different* VNets are isolated unless you explicitly connect them (peering, VPN, etc.).

**Subnets** divide a VNet into segments. Each subnet can have its own NSG (firewall rules).

```
┌── VNet: 10.0.0.0/16 ──────────────────────────────────────────────┐
│                                                                     │
│  ┌── Subnet: web (10.0.1.0/24) ─┐   ┌── Subnet: db (10.0.2.0/24) ─┐ │
│  │  🌐 Web Server VM             │   │  🗄️  Database VM              │ │
│  │  NSG: allow 80,443 from any   │   │  NSG: allow 5432 from web    │ │
│  │       allow 22 from Bastion   │   │       deny all else          │ │
│  └───────────────────────────────┘   └──────────────────────────────┘ │
│                                                                       │
│  ┌── Subnet: AzureBastionSubnet (10.0.3.0/26) ──┐                     │
│  │  🔒 Azure Bastion                            │                     │
│  └──────────────────────────────────────────────┘                     │
└───────────────────────────────────────────────────────────────────────┘
```

### Exam tip
- VNets provide **isolation** (different VNets can't talk by default).
- Subnets provide **segmentation** within a VNet.
- Both are **free** in Azure.

---
## 2. Network Security Groups (NSGs)

NSGs are **packet filters** — they allow or deny traffic based on source/destination IP, port, and protocol. They are applied to subnets or individual NICs.

Each NSG has a list of **rules** evaluated by priority (lowest number = highest priority). The first rule that matches wins — later rules are ignored.

### Bad practice → Best practice

| ❌ Bad | ✅ Best |
|-------|--------|
| `AllowRDPFromAny` (port 3389, source `*`) — invites brute-force attacks | `AllowSSHFromBastion` (port 22, source = Bastion subnet only) |
| One huge NSG on the VNet | One NSG per subnet, least-privilege rules |
| Rely on Azure defaults only | Explicit `DenyAllInbound` at 4096 |

Below we simulate both configurations and replay the same traffic against each one.

In [ ]:
from ipaddress import ip_address, ip_network

# A tiny NSG evaluator that behaves like Azure: rules are ordered by priority,
# first match wins, and an implicit Deny All Inbound lives at 65500.

def matches(rule, src_ip: str, dst_port: int) -> bool:
    # Port match (int or '*' wildcard)
    if rule['dst_port'] != '*' and rule['dst_port'] != dst_port:
        return False
    src = rule['src']
    if src == '*':
        return True
    if src == 'VirtualNetwork':
        # Azure's VirtualNetwork tag covers any RFC1918 address in the VNet.
        return ip_address(src_ip) in ip_network('10.0.0.0/16')
    # Otherwise treat `src` as a CIDR (e.g. '10.0.3.0/26') or single IP.
    try:
        return ip_address(src_ip) in ip_network(src, strict=False)
    except ValueError:
        return False


def evaluate_nsg(src_ip: str, dst_port: int, rules: list) -> dict:
    for rule in sorted(rules, key=lambda r: r['priority']):
        if matches(rule, src_ip, dst_port):
            return {
                'matched_rule': rule['name'],
                'priority': rule['priority'],
                'action': '✅ ALLOW' if rule['action'] == 'allow' else '🚫 DENY',
            }
    return {'matched_rule': 'implicit-deny', 'priority': 65500, 'action': '🚫 DENY'}


def replay(label: str, rules: list, traffic: list):
    print(f'=== {label} ===')
    for desc, src, port in traffic:
        r = evaluate_nsg(src, port, rules)
        print(f"  {desc:<42} → {r['action']}  (rule: {r['matched_rule']})")
    print()


TRAFFIC = [
    ('Internet attacker → RDP (3389)',   '203.0.113.9',  3389),
    ('Internet user → HTTPS (443)',      '203.0.113.1',  443),
    ('Bastion subnet → SSH (22)',        '10.0.3.5',     22),
    ('Internet user → SSH (22)',         '203.0.113.1',  22),
    ('Web subnet → DB (5432)',           '10.0.1.10',    5432),
]

# ❌ Bad NSG — port 3389 is exposed to the world
BAD_RULES = [
    {'priority': 100, 'name': 'AllowRDPFromAny', 'src': '*',           'dst_port': 3389, 'action': 'allow'},
    {'priority': 200, 'name': 'AllowHTTPS',      'src': '*',           'dst_port': 443,  'action': 'allow'},
    {'priority': 65500,'name': 'DenyAllInbound', 'src': '*',           'dst_port': '*',  'action': 'deny'},
]

# ✅ Best NSG — least-privilege, SSH only from Bastion
GOOD_RULES = [
    {'priority': 100, 'name': 'AllowHTTPS',          'src': '*',            'dst_port': 443,  'action': 'allow'},
    {'priority': 110, 'name': 'AllowHTTP',           'src': '*',            'dst_port': 80,   'action': 'allow'},
    {'priority': 200, 'name': 'AllowSSHFromBastion', 'src': '10.0.3.0/26',  'dst_port': 22,   'action': 'allow'},
    {'priority': 300, 'name': 'AllowDBFromWeb',      'src': '10.0.1.0/24',  'dst_port': 5432, 'action': 'allow'},
    {'priority': 400, 'name': 'DenyRDP',             'src': '*',            'dst_port': 3389, 'action': 'deny'},
    {'priority': 65000,'name': 'AllowVNetInbound',   'src': 'VirtualNetwork','dst_port': '*', 'action': 'allow'},
    {'priority': 65500,'name': 'DenyAllInbound',     'src': '*',            'dst_port': '*',  'action': 'deny'},
]

replay('❌ BAD NSG (RDP open to the internet)', BAD_RULES,  TRAFFIC)
replay('✅ BEST NSG (least privilege)',          GOOD_RULES, TRAFFIC)

### Key NSG facts for the exam

- Rules evaluated by **priority** (100-4096 for custom rules, lowest number wins).
- Default rules include `AllowVNetInbound`, `AllowAzureLoadBalancerInbound`, `DenyAllInbound`.
- Applied at **subnet** or **NIC** level. Both are evaluated — most restrictive wins.
- NSGs are **stateful** — if inbound is allowed, the response is automatically allowed.
- **Free** to use (no cost per NSG).

---
## 3. Azure Firewall — FQDN filtering for outbound traffic

NSGs work with IPs and ports but can't say *"allow traffic to `*.ubuntu.com` but nothing else"*. That's what **Azure Firewall** is for.

Below we simulate an Azure Firewall **application rule collection** that restricts a VM's outbound internet traffic to an allow-list of FQDNs.

In [ ]:
import fnmatch

# Simulated Azure Firewall application rules (FQDN-based, outbound)
APP_RULES = [
    {'name': 'AllowUbuntuMirrors', 'fqdns': ['*.ubuntu.com', 'archive.ubuntu.com'],    'action': 'allow'},
    {'name': 'AllowMicrosoft',     'fqdns': ['*.microsoft.com', '*.azure.com', '*.microsoftonline.com'], 'action': 'allow'},
    {'name': 'AllowGitHub',        'fqdns': ['github.com', '*.githubusercontent.com'], 'action': 'allow'},
]

def firewall_evaluate(fqdn: str) -> str:
    for rule in APP_RULES:
        for pattern in rule['fqdns']:
            if fnmatch.fnmatch(fqdn, pattern):
                return f"✅ ALLOW (rule: {rule['name']})"
    # Default Azure Firewall action is deny.
    return '🚫 DENY (default deny — threat intel / no match)'

outbound = [
    'archive.ubuntu.com',         # legit apt mirror
    'login.microsoftonline.com',  # Azure AD auth
    'raw.githubusercontent.com',  # legit code fetch
    'c2.evilcorp.ru',             # 🚨 malware C2
    'bitcoin-miner.pool.io',      # 🚨 cryptominer
]
print('=== Azure Firewall: outbound FQDN filtering ===')
for f in outbound:
    print(f'  {f:<32} → {firewall_evaluate(f)}')

### NSG vs Azure Firewall vs WAF — which to pick?

| Feature | NSG | Azure Firewall | WAF |
|---------|-----|---------------|------|
| **Layer** | L3/L4 (IP, port) | L3-L7 (includes FQDN, TLS) | L7 (HTTP only) |
| **Scope** | Subnet/NIC | Entire VNet (centralized) | Web apps (App Gateway / Front Door) |
| **FQDN filtering** | ❌ | ✅ (e.g. `*.microsoft.com`) | N/A |
| **OWASP protection** | ❌ | ❌ | ✅ (SQLi, XSS, etc.) |
| **Threat intelligence** | ❌ | ✅ (block known malicious IPs) | ✅ |
| **Cost** | Free | ~$900/month+ | Included with App Gateway |
| **Use case** | Micro-segmentation | Central egress/ingress control | Protect web APIs |

> 💡 These are **complementary**, not alternatives. A well-designed network uses all three.

---
## 4. Web Application Firewall (WAF)

A WAF inspects **HTTP requests** and blocks attacks that match OWASP patterns — SQL injection, cross-site scripting, path traversal, etc.

Here we run a tiny WAF simulator with a handful of OWASP Core Rule Set-style patterns against a mix of benign and malicious requests.

In [ ]:
import re

# A very simplified version of what the OWASP CRS does.
WAF_RULES = [
    ('SQL injection', re.compile(r"(?i)(\b(union|select|or|and)\b.*(=|--|;)|'\s*or\s*'1'='1)")),
    ('XSS',           re.compile(r"(?i)(<script|onerror\s*=|javascript:)")),
    ('Path traversal',re.compile(r"\.\./|\.\.\\")),
    ('Command inj.',  re.compile(r";\s*(cat|ls|whoami|rm)\b|\|\s*nc\b")),
]

def waf_inspect(request: str) -> str:
    for name, pattern in WAF_RULES:
        if pattern.search(request):
            return f'🚫 BLOCK — matched {name} rule'
    return '✅ ALLOW'

requests = [
    "GET /api/users?id=42",                                        # benign
    "GET /api/users?id=1' OR '1'='1",                              # SQLi
    "POST /comment body=<script>alert(1)</script>",                # XSS
    "GET /download?file=../../etc/passwd",                         # path traversal
    "GET /ping?host=8.8.8.8;cat /etc/passwd",                      # command injection
    "GET /products?category=books",                                # benign
]
print('=== Web Application Firewall (L7) ===')
for r in requests:
    print(f'  {r[:55]:<55} → {waf_inspect(r)}')

### Two WAF modes to remember
- **Detection mode** — log attacks but don't block (useful when first rolling out a WAF).
- **Prevention mode** — block matching requests with an HTTP `403`.

---
## 5. Azure DDoS Protection

Two tiers — know the difference for the exam:

| | DDoS Network Protection (Basic) | DDoS Network Protection (Standard) |
|-|------------|----------|
| **Cost** | Free (on by default) | ~$2,944/month |
| **Scope** | All Azure services | Per-VNet opt-in |
| **Features** | Automatic traffic monitoring | Adaptive tuning, attack analytics, rapid response team, cost protection |

> **Exam tip**: Basic protection is *always on* for all Azure resources. Standard adds tuning, alerting, and the DDoS Rapid Response (DRR) team.

---
## 6. Azure Bastion — RDP/SSH without public IPs

### Bad practice → Best practice

| ❌ Bad | ✅ Best |
|-------|--------|
| VM has a public IP and NSG rule `AllowRDPFromAny` on port 3389 | VM has **no public IP**; admin connects via Bastion over HTTPS in the browser |
| Credentials travel over the raw internet | Traffic stays inside Azure; audit logs in the portal |
| Everyone on the internet can attempt brute-force | Only Azure AD users with RBAC on Bastion can connect |

```
Admin → Azure Portal (HTTPS/TLS 443) → Bastion → VM (private IP only)
```

- No public IP needed on VMs.
- No NSG rules for RDP (3389) or SSH (22) from the internet.
- Requires a dedicated subnet named `AzureBastionSubnet` (/26 or larger).

In [ ]:
# Compare attack surface: direct RDP vs Bastion
scenarios = [
    {
        'name': '❌ VM with public IP + RDP open',
        'public_ip_on_vm': True,
        'exposed_ports': [3389],
        'auth_path':     'Internet → VM (NTLM/Kerberos)',
    },
    {
        'name': '✅ Private VM + Azure Bastion',
        'public_ip_on_vm': False,
        'exposed_ports': [],
        'auth_path':     'Internet → Azure AD → Bastion (TLS) → VM',
    },
]

internet_attacker = {'source': '203.0.113.66', 'target_port': 3389}

for s in scenarios:
    reachable = s['public_ip_on_vm'] and internet_attacker['target_port'] in s['exposed_ports']
    status = '🚨 REACHABLE from internet' if reachable else '🛡️  NOT reachable from internet'
    print(f"{s['name']}")
    print(f"  Auth path : {s['auth_path']}")
    print(f"  Attacker  : {status}\n")

---
## Summary

| Service | Purpose | Key exam fact |
|---------|---------|---------------|
| **VNet** | Network isolation | Different VNets are isolated by default |
| **NSG** | Subnet/NIC packet filter | Rules by priority, stateful, free |
| **Azure Firewall** | Centralized L3-L7 filtering | FQDN filtering, threat intel |
| **WAF** | Protect web apps from OWASP | SQLi, XSS protection; detection vs prevention mode |
| **DDoS Protection** | Absorb volumetric attacks | Basic is free and always on |
| **Bastion** | Secure RDP/SSH without public IPs | Requires `AzureBastionSubnet` |

**Next**: [Notebook 2 — Key Vault and Defender for Cloud](02_key_vault_and_defender.ipynb)